# ATMS GPROF-NN 1D retrievals

This notebook assess GPROF-NN 1D retrieval based on Satformer simulations.

In [ ]:
%load_ext autoreload
%autoreload 2
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr

In [ ]:
from gprof_nn.plotting import set_style
set_style()

In [ ]:
import pansat
pansat.__file__

In [ ]:
stats = xr.load_dataset("/gdata1/simon/gprof_v8/models/atms/gprof_nn_1d_sf/stats/input/brightness_temperatures.nc")
tb_bins = stats.bin_boundaries

In [ ]:
from  tqdm import tqdm

GPROF_CHANS = [8, 10, 12, 13, 14]
counts_input = np.zeros((5, tb_bins.shape[-1] - 1))
input_files = sorted(list(Path("/edata1/simon/gprof_v8/collocations/combined/atms/on_swath/").glob("*.nc")))
for path in tqdm(input_files[:1000]):
    with xr.open_dataset(path, group="input_data") as data:
        tbs = data.observations_gprof.data
        for chan in range(5):
            counts_input[chan] += np.histogram(tbs[..., chan], tb_bins[GPROF_CHANS[chan]])[0]

In [ ]:
chan = 4
x = tb_bins[GPROF_CHANS[chan]]
x = 0.5 * (x[1:] + x[:-1])
plt.plot(x, counts_input[chan] / counts_input[chan].sum(), label="Input")
plt.plot(x, stats.counts[GPROF_CHANS[chan]] / stats.counts[GPROF_CHANS[chan]].sum(), label="Training")
plt.legend()

## Ancillary data

In [ ]:
stats = xr.load_dataset("/gdata1/simon/gprof_v8/models/atms/gprof_nn_1d_sf/stats/input/ancillary_data.nc")
bins = stats.bin_boundaries

In [ ]:
from  tqdm import tqdm

ANCILLARY_VARIABLES = [
    "land_fraction",
    "mountain_type",
    "elevation",
    "two_meter_temperature",
    "total_column_water_vapor",
    "orographic_wind",
    "10m_wind",
    "moisture_convergence",
    "convective_precipitation",
    "leaf_area_index",
    "ice_fraction",
]

mins = {}
maxs = {}

counts_input = np.zeros((11, tb_bins.shape[-1] - 1))
input_files = sorted(list(Path("/edata1/simon/gprof_v8/collocations/combined/atms/on_swath/").glob("*.nc")))
for path in tqdm(input_files[:1_000]):
    with xr.open_dataset(path, group="input_data") as data:
        for chan in range(11):
            anc = data[ANCILLARY_VARIABLES[chan]].data
            counts_input[chan] += np.histogram(anc, bins[chan])[0]
            var = ANCILLARY_VARIABLES[chan]
            mins[var] = min(mins.get(var, 9999), np.nanmin(anc))
            maxs[var] = max(mins.get(var, -9999), np.nanmax(anc))

In [ ]:
stats["min"].data

In [ ]:
m

In [ ]:
chan = 10
x = tb_bins[chan]
x = 0.5 * (x[1:] + x[:-1])
plt.plot(x, counts_input[chan] / counts_input[chan].sum(), label="Input")
plt.plot(x, stats.counts[chan] / stats.counts[chan].sum(), label="Training")
plt.legend()

## Data

We use collocation between ATMS and GPM CMB and match them with the corresponding GPROF simulator files.


In [ ]:
collocations = sorted(list(Path("/edata1/simon/gprof_v8/collocations/combined/atms/gridded/").glob("*.nc")))

In [ ]:
import hdf5plugin
from pyresample.geometry import SwathDefinition

colloc_ind = 224
atms_observations = xr.load_dataset(collocations[colloc_ind], group="input_data")
reference_data = xr.load_dataset(collocations[colloc_ind], group="reference_data")

def get_granule(reference_data: xr.Dataset) -> int:
    """
    Extract granule from collocation attributes.

    Args:
        reference_data: An xarray.Dataset containing the collocation reference data

    Return:
        The granule number as an integer.
    """
    l1c_file = reference_data.attrs["input_files"]
    granule = int(l1c_file.split(".")[-3])
    return granule

granule = get_granule(reference_data)
lons, lats = np.meshgrid(atms_observations.longitude.data, atms_observations.latitude.data)
area = SwathDefinition(lats=lats, lons=lons)

In [ ]:
def load_retrieval_results(base_path: Path, collocation_file):
    """
    Load GPROF-NN 1D retrieval and resample to grid.
    """
    base_path = Path(base_path)
    return xr.load_dataset(base_path / collocation_file.name)

In [ ]:
collocations[0]

In [ ]:
gprof_nn_1d_sf_path = Path("/edata1/simon/gprof_v8/results/collocations/atms/gprof_nn_1d_sf/")
gprof_nn_1d_sf_noise_path = Path("/edata1/simon/gprof_v8/results/collocations/atms/gprof_nn_1d_sf_noise/")
gprof_nn_1d_path = Path("/edata1/simon/gprof_v8/results/collocations/atms/gprof_nn_1d/")
res = load_retrieval_results(gprof_nn_1d_sf_path, collocations[0])

In [ ]:
from datetime import datetime
from pansat.products.satellite.gpm import l2a_clim_gprof_noaa20_atms
def load_gprof_results(collocation_file):
    date = datetime.strptime(collocation_file.name.split("_")[-1], "%Y%m%d%H%M%S.nc")
    with xr.open_dataset(collocation_file, group="input_data") as data:
        scan_start = data.attrs["scan_start"]
        scan_end = data.attrs["scan_end"]
        gprof_recs = l2a_clim_gprof_noaa20_atms.get(date)
        gprof_data = l2a_clim_gprof_noaa20_atms.open(gprof_recs[0])[{"scans": slice(scan_start, scan_end)}]
    return gprof_data


## gprof_data = load_gprof_results(collocations[0])

In [ ]:
from gprof_nn.data.sim import SimFile
from pansat.utils import resample_data
from scipy.interpolate import interpn

from gprof_nn import sensors
from gprof_nn.data.sim import collocate_targets, SimulatorInput
from gprof_nn.data.sim import simulate_tbs_satformer
from gprof_nn.data.utils import decompress_scene


def find_sim_file(granule: int) -> Path:
    """
    Find the sim file for a given granule.

    Args:
        granule: The granule number as an integer

    Return:
        A path object pointing to the sim file corresopnding to the given granule.
    """
    pattern = f"**/*.{granule:06}.sim"
    sim_files = sorted(list(Path("/qdata1/pbrown/dbaseV8/simV8x_atms/").glob(pattern)))
    return next(iter(sim_files))

def load_and_resample_sim_data(
    granule,
    atms_observations,
    area,
):
    """
    Load and resample simulator data for a given granule and resample to the given geometry.

    Args:
        granule: The granule number as an integer.
        atms_observations: The collocated ATMS observations to use to interpolate the simulated brightness temperatures.
        area: A pyresample area definition defining the grid to which to resample the simulator data.

    Return:
        An xarray.Dataset containing the resample simulator data.
    """
    sim_file_path = find_sim_file(granule)
    sim_file = SimFile(sim_file_path)

    sim_data = sim_file.to_xarray_dataset()
    
    targets = [ "latitude", "longitude", "surface_precip", "surface_type"]
    sim_data = decompress_scene(sim_data, targets)
    
    sim_data_r = resample_data(sim_data, area, new_dims=("latitude", "longitude"), radius_of_influence=15e3)
    angles = sim_file.header[0][3]
    sim_data_r = sim_data_r.assign_coords(angles=angles).sortby("angles")
    
    lons, lats = np.meshgrid(sim_data_r.longitude.data, sim_data_r.latitude.data, indexing="xy")
    ang = atms_observations.earth_incidence_angle.data[..., 0]
    pts_out = np.stack((lats, lons, ang), axis=-1)
    pts_in = (sim_data_r.latitude.data, sim_data_r.longitude.data, sim_data_r.angles.data)
    surface_precip = sim_data_r.surface_precip.data
    surface_precip_i = interpn(pts_in, surface_precip, pts_out, bounds_error=False, fill_value=np.nan)
    sim_data_r["surface_precip"] = (("latitude", "longitude"), surface_precip_i)
    
    return sim_data_r

In [ ]:
sim_files = sorted(list(Path("/qdata1/pbrown/dbaseV8/simV8x_atms/").glob("**/*.sim")))

## Case study

In [ ]:
colloc_ind = 22
colloc = collocations[colloc_ind]

atms_observations = xr.load_dataset(colloc, group="input_data")
lons, lats = np.meshgrid(atms_observations.longitude.data, atms_observations.latitude.data)
area = SwathDefinition(lats=lats, lons=lons)

reference_data = xr.load_dataset(colloc, group="reference_data")
granule = get_granule(reference_data)
    
sim_data_r = load_and_resample_sim_data(granule, atms_observations, area)
gprof_nn_1d_sf_r = resample_data(load_retrieval_results(gprof_nn_1d_sf_path, colloc), area, radius_of_influence=60e3)
gprof_nn_1d_sf_noise_r = resample_data(load_retrieval_results(gprof_nn_1d_sf_noise_path, colloc), area, radius_of_influence=60e3)
gprof_nn_1d_r = resample_data(load_retrieval_results(gprof_nn_1d_path, colloc), area, radius_of_influence=60e3)
gprof = resample_data(load_gprof_results(colloc), area, radius_of_influence=60e3)

In [ ]:
import cartopy.crs as ccrs
from matplotlib.gridspec import GridSpec

fig = plt.figure(figsize=(16, 8))
gs = GridSpec(1, 3)
crs = ccrs.PlateCarree()

ax = fig.add_subplot(gs[0, 0], projection=crs)

lons = sim_data_r.longitude.data
lats = sim_data_r.latitude.data
sp_sim = sim_data_r.surface_precip.data
ax.pcolormesh(lons, lats, sp_sim, vmin=0, vmax=1)
ax.coastlines(color="grey")

ax = fig.add_subplot(gs[0, 1], projection=crs)
lons = gprof.longitude.data
lats = gprof.latitude.data
sp = gprof.surface_precipitation.data.copy()
sp[sp < 0] = np.nan
ax.pcolormesh(lons, lats, sp, vmin=0, vmax=1)
ax.contour(lons, lats, np.isfinite(sp_sim), colors=["grey"], levels=[0.5])
ax.coastlines(color="grey")

valid = np.isfinite(sp) * np.isfinite(sp_sim)
print(np.mean(sp[valid] - sp_sim[valid]), np.corrcoef(sp[valid], sp_sim[valid])[0, 1])


ax = fig.add_subplot(gs[0, 2], projection=crs)
lons = gprof_nn_1d_r.longitude.data
lats = gprof_nn_1d_r.latitude.data
sp = gprof_nn_1d_sf_r.surface_precip.data.copy()
ax.pcolormesh(lons, lats, sp, vmin=0, vmax=1)
ax.contour(lons, lats, np.isfinite(sp_sim), colors=["grey"], levels=[0.5], vmin=0, vmax=1)
ax.coastlines(color="grey")

print(np.mean(sp[valid] - sp_sim[valid]), np.corrcoef(sp[valid], sp_sim[valid])[0, 1])

In [ ]:
plt.pcolormesh(lons, lats, sp, vmin=0, vmax=3)
plt.contour(lons, lats, np.isfinite(sp_sim), colors=["grey"], levels=[0.5])
plt.show()

In [ ]:
plt.pcolormesh(lons, lats, atms_observations["observations_gprof"][..., 4])
plt.xlim(-45, -30)
plt.ylim(45, 55)

In [ ]:
import cartopy.crs as ccrs
from matplotlib.gridspec import GridSpec

fig = plt.figure(figsize=(16, 4))
gs = GridSpec(1, 3)
crs = ccrs.PlateCarree()

ax = fig.add_subplot(gs[0, 0], projection=crs)

lons = sim_data_r.longitude.data
lats = sim_data_r.latitude.data
sp_sim = sim_data_r.surface_precip.data
ax.pcolormesh(lons, lats, sp_sim, vmin=0, vmax=1)
ax.coastlines(color="grey")

ax = fig.add_subplot(gs[0, 1], projection=crs)
lons = gprof.longitude.data
lats = gprof.latitude.data
sp = gprof.surface_precipitation.data.copy()
sp[sp < 0] = np.nan
ax.pcolormesh(lons, lats, sp, vmin=0, vmax=1)
ax.contour(lons, lats, np.isfinite(sp_sim), colors=["grey"], levels=[0.5])
ax.coastlines(color="grey")

valid = np.isfinite(sp) * np.isfinite(sp_sim)
print(np.mean(sp[valid] - sp_sim[valid]), np.corrcoef(sp[valid], sp_sim[valid])[0, 1])


ax = fig.add_subplot(gs[0, 2], projection=crs)
lons = gprof_nn_1d_r.longitude.data
lats = gprof_nn_1d_r.latitude.data
sp = gprof_nn_1d_r.surface_precip.data.copy()
ax.pcolormesh(lons, lats, sp, vmin=0, vmax=1)
ax.contour(lons, lats, np.isfinite(sp_sim), colors=["grey"], levels=[0.5])
ax.coastlines(color="grey")

print(np.mean(sp[valid] - sp_sim[valid]), np.corrcoef(sp[valid], sp_sim[valid])[0, 1])

In [ ]:
import cartopy.crs as ccrs
from matplotlib.gridspec import GridSpec

fig = plt.figure(figsize=(16, 4))
gs = GridSpec(1, 3)
crs = ccrs.PlateCarree()

ax = fig.add_subplot(gs[0, 0], projection=crs)

lons = sim_data_r.longitude.data
lats = sim_data_r.latitude.data
sp_sim = sim_data_r.surface_precip.data
ax.pcolormesh(lons, lats, sp_sim)

ax = fig.add_subplot(gs[0, 1], projection=crs)
lons = gprof.longitude.data
lats = gprof.latitude.data
sp = gprof.surface_precipitation.data.copy()
sp[sp < 0] = np.nan
ax.pcolormesh(lons, lats, sp)

valid = np.isfinite(sp) * np.isfinite(sp_sim)
print(np.mean(sp[valid] - sp_sim[valid]), np.corrcoef(sp[valid], sp_sim[valid])[0, 1])


ax = fig.add_subplot(gs[0, 2], projection=crs)
lons = gprof_nn_1d_r.longitude.data
lats = gprof_nn_1d_r.latitude.data
sp = gprof_nn_1d_r.surface_precip.data.copy()
ax.pcolormesh(lons, lats, sp)

print(np.mean(sp[valid] - sp_sim[valid]), np.corrcoef(sp[valid], sp_sim[valid])[0, 1])

In [ ]:
import cartopy.crs as ccrs
from matplotlib.gridspec import GridSpec

fig = plt.figure(figsize=(16, 4))
gs = GridSpec(1, 3)
crs = ccrs.PlateCarree()

ax = fig.add_subplot(gs[0, 0], projection=crs)

lons = sim_data_r.longitude.data
lats = sim_data_r.latitude.data
sp_sim = sim_data_r.surface_precip.data
ax.pcolormesh(lons, lats, sp_sim)

ax = fig.add_subplot(gs[0, 1], projection=crs)
lons = gprof.longitude.data
lats = gprof.latitude.data
sp = gprof.surface_precipitation.data.copy()
sp[sp < 0] = np.nan
ax.pcolormesh(lons, lats, sp)

valid = np.isfinite(sp) * np.isfinite(sp_sim)
print(np.mean(sp[valid] - sp_sim[valid]), np.corrcoef(sp[valid], sp_sim[valid])[0, 1])


ax = fig.add_subplot(gs[0, 2], projection=crs)
lons = gprof_nn_1d_r.longitude.data
lats = gprof_nn_1d_r.latitude.data
sp = gprof_nn_1d_r.surface_precip.data.copy()
ax.pcolormesh(lons, lats, sp)

print(np.mean(sp[valid] - sp_sim[valid]), np.corrcoef(sp[valid], sp_sim[valid])[0, 1])

In [ ]:
plt.pcolormesh(gprof_nn_1d_r.surface_precip.data)

## Evaluate multiple scenes


## Collect data from collocations

Below, we iterate over all collocations and extract data from pixels with valid simulator data.

In [ ]:
inpt = xr.load_dataset(colloc, group="input_data")

In [ ]:
stats = xr.load_dataset("/gdata1/simon/gprof_v8/models/atms/gprof_nn_1d_sf/stats/input/ancillary_data.nc")
stats["min"]

In [ ]:
plt.pcolormesh(inpt.observations_gprof[..., -5])
plt.colorbar()

In [ ]:
eia[-1].shape, sp_ref[-1].shape

In [ ]:
from tqdm import tqdm
sp_ref = []
sp_gprof = []
sp_gprof_nn_1d_sf = []
eia = []
surface_type = []

scene_ind = 0
for colloc in tqdm(collocations[:1000]):

    try:
        atms_observations = xr.load_dataset(colloc, group="input_data")
        lons, lats = np.meshgrid(atms_observations.longitude.data, atms_observations.latitude.data)
        area = SwathDefinition(lats=lats, lons=lons)
    
        reference_data = xr.load_dataset(colloc, group="reference_data")
        granule = get_granule(reference_data)
        
        sim_data_r = load_and_resample_sim_data(granule, atms_observations, area)
        gprof_nn_1d_data = resample_data(load_retrieval_results(gprof_nn_1d_path, colloc), area, radius_of_influence=60e3)
        gprof_data = resample_data(load_gprof_results(colloc), area, radius_of_influence=60e3)
        valid = (
            (sim_data_r.surface_precip >= -10) *
            (gprof_data.surface_precipitation.data >= -10)
        )
        sp_ref.append(sim_data_r["surface_precip"].data[valid])
        sp_gprof.append(gprof_data["surface_precipitation"].data[valid])
        sp_gprof_nn_1d_sf.append(gprof_nn_1d_data["surface_precip"].data[valid])
        surface_type.append(atms_observations["surface_type"].data[valid])
        eia.append(atms_observations["earth_incidence_angle_gprof"].data[valid])
        sp = gprof_nn_1d_data["surface_precip"].data[valid]

        sp_ref_i = sp_ref[-1]
        sp_nn_sf_i = sp_gprof_nn_1d_sf[-1]
        sfc = surface_type[-1]

        scene_ind += 1
        print("REL BIAS :: ", scene_ind, (sp_nn_sf_i - sp_ref_i).mean() / sp_ref_i.mean(), sp_ref_i.mean())
        
        if sp.max() > 1_000:
            print(colloc)
            
    except Exception:
        pass


In [ ]:
results = xr.Dataset({
    "sp_ref":  (("samples"), np.concatenate(sp_ref)),
    "sp_gprof": (("samples"), np.concatenate(sp_gprof)),
    "sp_gprof_nn_sf": (("samples"), np.concatenate(sp_gprof_nn_1d_sf)),
    "surface_type": ("samples", np.concatenate(surface_type)),
    "eia": ("samples", np.concatenate(eia)),
})


In [ ]:
results = results[{"samples": np.isfinite(results.sp_gprof_nn_sf.data)}]

In [ ]:
results.sp_gprof_nn_sf.data.max()

In [ ]:
import pandas as pd

def calculate_error_stats(results):
    sp_ref = results.sp_ref.data
    sp_gprof = results.sp_gprof.data
    sp_nn_sf = results.sp_gprof_nn_sf.data
    bias_gprof = 100.0 * (sp_gprof - sp_ref).mean() / sp_ref.mean()
    bias_nn_sf = 100.0 * (sp_nn_sf - sp_ref).mean() / sp_ref.mean()
    corr_gprof = np.corrcoef(sp_ref, sp_gprof)[0, 1]
    corr_nn_sf = np.corrcoef(sp_ref, sp_nn_sf)[0, 1]
    mse_gprof = np.mean((sp_ref - sp_gprof)**2)
    mse_nn_sf = np.mean((sp_ref - sp_nn_sf)**2)
    mae_gprof = np.mean(np.abs(sp_ref - sp_gprof))
    mae_nn_sf = np.mean(np.abs(sp_ref - sp_nn_sf))
    return pd.DataFrame({
        "Algorithm": ["GPROF", "GPROF-NN 1D (SF)"],
        "Bias": [bias_gprof, bias_nn_sf],
        "MSE": [mse_gprof, mse_nn_sf],
        "MAE": [mae_gprof, mae_nn_sf],
        "Correlation": [corr_gprof, corr_nn_sf],
    })
    

In [ ]:
results = results[{"samples": results.sp_gprof_nn_sf.data < 1_000}]
results.to_netcdf("results_gprofnn.nc")

In [ ]:
results = xr.load_dataset("results_gprofnn.nc")

In [ ]:
results_ocean = results[{"samples": results.surface_type.data == 1}]
calculate_error_stats(results_ocean)

In [ ]:
results_ocean = results[{"samples": results.surface_type.data == 1}]
calculate_error_stats(results_ocean)

In [ ]:
results_veg = results[{"samples": (2 < results.surface_type.data) * (results.surface_type.data < 8)}]
calculate_error_stats(results_veg)

In [ ]:
results_veg = results[{"samples": (2 < results.surface_type.data) * (results.surface_type.data < 8)}]
calculate_error_stats(results_veg)

In [ ]:
results_veg = results[{"samples": (11 < results.surface_type.data) * (results.surface_type.data < 16)}]
calculate_error_stats(results_veg)

In [ ]:
results_veg = results[{"samples": (11 < results.surface_type.data) * (results.surface_type.data < 16)}]
calculate_error_stats(results_veg)

In [ ]:
np.mean(results_ocean.sp_gprof.data - results_ocean.sp_ref.data) / results_ocean.sp_ref.data.mean()

In [ ]:
np.corrcoef(results_ocean.sp_ref.data, results_ocean.sp_gprof.data)[0, 1]

In [ ]:
np.corrcoef(results_ocean.sp_ref.data, results_ocean.sp_gprof_nn_sf.data)[0, 1]

In [ ]:
results_ocean.sp_gprof_nn_sf.data.mean()

In [ ]:
results_ocean.sp_gprof.data.mean()

In [ ]:
results_ocean.sp_ref.data.mean()

In [ ]:
((results_ocean.sp_gprof.data - results_ocean.sp_ref.data)**2).mean()

In [ ]:
((results_ocean.sp_gprof_nn_sf.data - results_ocean.sp_ref.data)**2).mean()

In [ ]:
from scipy.stats import binned_statistic
results.eia.data.max()
bins = np.linspace(0, 50, 6)
va_mean_sf = binned_statistic(results_ocean.eia.data, results_ocean.sp_gprof_nn_sf.data)[0]
va_mean_gprof = binned_statistic(results_ocean.eia.data, results_ocean.sp_gprof.data)[0]
va_mean_ref = binned_statistic(results_ocean.eia.data, results_ocean.sp_ref.data)[0]


In [ ]:
plt.plot(va_mean_sf / va_mean_ref, label="SF")
#plt.plot(va_mean_gprof, label="GPROF V7")
#plt.plot(va_mean_ref, label="REF")
plt.legend()

In [ ]:
plt.plot(va_mean_sf, label="SF")
plt.plot(va_mean_gprof, label="GPROF V7")
plt.plot(va_mean_ref, label="REF")
plt.legend()

In [ ]:
stats = xr.load_dataset("/gdata1/simon/gprof_v8/models/atms/gprof_nn_1d_sf/stats/input/viewing_angles.nc")


In [ ]:
stats["max"]

In [ ]:
plt.plot(stats["counts"][0])

In [ ]:
np.mean(results_ocean.sp_gprof_nn_sf.data ) / results_ocean.sp_ref.data.mean()

In [ ]:
from matplotlib.gridspec import GridSpec

CHANNELS = [
    "89 GHz",
    "164 GHz",
    "183 +/- 1 GHz",
    "183 +/- 3 GHz",
    "183 +/- 7 GHz",
]

def make_scater_plots(results):
    fig = plt.figure(figsize=(20, 25))
    gs = GridSpec(5, 5, width_ratios=[0.3, 1.0, 1.0, 1.0, 1.0])
    
    
    for chan in range(5):
        
        ax = fig.add_subplot(gs[chan, 0])
        ax.set_axis_off()
        ax.text(0, 0, CHANNELS[chan], rotation=90, ha="center", va="center")
        ax.set_ylim(-2, 2)
        
        tbs_act = results["tbs_actual"].data[..., chan]
        tbs_sim = results["tbs_sim"].data[..., chan]
        tbs_bias = results["tbs_sim_bias"].data[..., chan]
        tbs_sf = results["tbs_sf"].data[..., chan]
        eia = results["eia"].data
        
        #
        # Simulated TBS
        #
        
        ax = fig.add_subplot(gs[chan, 1])
        if chan == 0:
            ax.set_title("Simulated", loc="center")
        bins = np.linspace(tbs_act.min(), tbs_act.max())
        tbs = tbs_sim
        dens = np.histogram2d(tbs_act, tbs, bins=bins)[0]
        dens /= dens.sum(1, keepdims=True)
        x = 0.5 * (bins[1:] + bins[:-1])
        ax.pcolormesh(x, x, dens.T)
        ax.plot(x, x, ls="--", c="grey")
        ax.set_aspect(1.0)
        
        bias = 100.0 * (tbs - tbs_act).mean() / np.mean(tbs_act)
        rmse = np.sqrt(np.mean((tbs - tbs_act) ** 2))
        corr = np.corrcoef(tbs_act, tbs)[0, 1]
        ax.text(0.1, 0.7, f"Bias:  {bias:.2f} %\nRMSE:  {rmse:.2f}\nCorr.: {corr:.2f}", transform=ax.transAxes, color="grey")

        ax.set_ylabel("Simulated $T_b$ [K]")
        ax.set_xlabel("Actual $T_b$ [K]")
        ax.grid(False)
        
        #
        # Bias-corrected, simulated TBs
        #
        
        ax = fig.add_subplot(gs[chan, 2])
        if chan == 0:
            ax.set_title("Simulated - Bias", loc="center")
        tbs = tbs_sim - tbs_bias
        dens = np.histogram2d(tbs_act, tbs, bins=bins)[0]
        dens /= dens.sum(1, keepdims=True)
        x = 0.5 * (bins[1:] + bins[:-1])
        ax.pcolormesh(x, x, dens.T)
        ax.plot(x, x, ls="--", c="grey")
        ax.set_aspect(1.0)
        
        bias = 100.0 * (tbs - tbs_act).mean() / np.mean(tbs_act)
        rmse = np.sqrt(np.mean((tbs - tbs_act) ** 2))
        corr = np.corrcoef(tbs_act, tbs)[0, 1]
        ax.text(0.1, 0.7, f"Bias:  {bias:.2f} %\nRMSE:  {rmse:.2f}\nCorr.: {corr:.2f}", transform=ax.transAxes, color="grey")
        
        ax.set_yticklabels([])
        ax.set_xlabel("Actual $T_b$ [K]")
        ax.grid(False)
        
        #
        # EIA adapted bias correction
        #
        
        ax = fig.add_subplot(gs[chan, 3])
        if chan == 0:
            ax.set_title(r"Simulated - $\frac{\cos(\theta_{\text{GMI}})}{\cos(\theta)}$ Bias", loc="center")
        tbs = tbs_sim - np.cos(np.deg2rad(48.0)) / np.cos(np.deg2rad(eia)) * tbs_bias
        dens = np.histogram2d(tbs_act, tbs, bins=bins)[0]
        dens /= dens.sum(1, keepdims=True)
        x = 0.5 * (bins[1:] + bins[:-1])
        ax.pcolormesh(x, x, dens.T)
        ax.plot(x, x, ls="--", c="grey")
        ax.set_aspect(1.0)
        
        bias = 100.0 * (tbs - tbs_act).mean() / np.mean(tbs_act)
        rmse = np.sqrt(np.mean((tbs - tbs_act) ** 2))
        corr = np.corrcoef(tbs_act, tbs)[0, 1]
        ax.text(0.1, 0.7, f"Bias:  {bias:.2f} %\nRMSE:  {rmse:.2f}\nCorr.: {corr:.2f}", transform=ax.transAxes, color="grey")
        
        ax.set_yticklabels([])
        ax.set_xlabel("Actual $T_b$ [K]")
        ax.grid(False)
        
        #
        # Satformer results
        #
        
        ax = fig.add_subplot(gs[chan, 4])
        if chan == 0:
            ax.set_title(r"Satformer", loc="center")
        
        tbs = tbs_sf
        dens = np.histogram2d(tbs_act, tbs, bins=bins)[0]
        dens /= dens.sum(1, keepdims=True)
        x = 0.5 * (bins[1:] + bins[:-1])
        ax.pcolormesh(x, x, dens.T)
        ax.plot(x, x, ls="--", c="grey")
        ax.set_aspect(1.0)
                                 
        bias = 100.0 * (tbs - tbs_act).mean() / np.mean(tbs_act)
        rmse = np.sqrt(np.mean((tbs - tbs_act) ** 2))
        corr = np.corrcoef(tbs_act, tbs)[0, 1]
        #ax.text(0.1 * x[0], 0.9 * x[-1], f"Bias:  {bias:.2f}\nRMSE:  {rmse:.2f}\nCorr.: {corr:.2f}")
        ax.text(0.1, 0.7, f"Bias:  {bias:.2f} %\nRMSE:  {rmse:.2f}\nCorr.: {corr:.2f}", transform=ax.transAxes, color="grey")
        
        ax.set_yticklabels([])
        ax.set_xlabel("Actual $T_b$ [K]")
        ax.grid(False)

    return fig, ax

In [ ]:
100

In [ ]:
fig, ax = make_scater_plots(results)

In [ ]:
fig, ax = make_scater_plots(results)

In [ ]:
fig, ax = make_scater_plots(results)

In [ ]:
fig, ax = make_scater_plots(results)

In [ ]:
results_h = results[{"samples": np.abs(results.eia) > 40}]
make_scater_plots(results_h)

In [ ]:
results_h = results[{"samples": np.abs(results.eia) > 40}]
make_scater_plots(results_h)

## Test simulations

In [ ]:
time_range = TimeRange("2020-06-03", "2020-06-04")
gmi_recs = l1c_r_gpm_gmi.get(time_range=time_range)
atms_recs = l1c_noaa20_atms.get(time_range=time_range)
gmi_index = Index.index(l1c_r_gpm_gmi, gmi_recs)
atms_index = Index.index(l1c_noaa20_atms, atms_recs)
matches = find_matches(gmi_index, atms_index)
input_loader = InputLoader(matches)

In [ ]:
inpt, fname, aux = input_loader.load_data(1)

In [ ]:
inpt["observations"].shape

In [ ]:
plt.pcolormesh(inpt["observations"][0, 2, 6])

In [ ]:
inpt.keys()

In [ ]:
tile = {name: tensor[..., 350:478, 20:148] for name, tensor in inpt.items()}
for name, tensor in inpt.items():
    if name.endswith("_mask"):
        tile[name] = inpt[name]
tbs_targ = aux["target_observations"].data[..., 64:128, 64:128]

In [ ]:
tile["observations"].shape

In [ ]:
plt.imshow(tile["observations"][0, 2, 0])

In [ ]:
import torch

with torch.no_grad():
    y_pred = model(tile)["output_observations"]
    y_pred = [tensor.expected_value().cpu().numpy().squeeze() for tensor in y_pred]

In [ ]:
from copy import deepcopy

props = tile["output_observation_props"].clone()
props[:, :, 1:] = props[:, :, :1]
props[0, 3, :] = torch.tensor(np.linspace(1.75, 5.2, 9))[..., None, None]
props[0, 4, :] = torch.tensor(np.linspace(416e3, 800e3, 9))[..., None, None]

tile_bw = deepcopy(tile)
tile_bw["output_observation_props"] = props

In [ ]:

with torch.no_grad():
    y_pred = model(tile_bw)["output_observations"]
    y_pred = [tensor.expected_value().cpu().numpy().squeeze() for tensor in y_pred]

## Beam width

In [ ]:
def simulate_beam_width(beam_width):
    
    props = tile["output_observation_props"].clone()[:, :, :1]
    props[0, 3, :] = beam_width
    tile_bw = deepcopy(tile)
    tile_bw["output_observation_props"] = props

    with torch.no_grad():
        y_pred = model(tile_bw)["output_observations"]
        y_pred = [tensor.expected_value().cpu().numpy().squeeze() for tensor in y_pred]

    plt.figure(figsize=(6.5, 5))
    plt.title(f"89 GHz, beam width = {beam_width:.2f} deg.")
    plt.imshow(y_pred[0], vmin=160, vmax=240)
    plt.colorbar(label="$T_b$ [K]")
    plt.grid(False)


In [ ]:
model

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import interact

interact(simulate_beam_width, beam_width=(1.5, 5.2))


## WV-channel offset

In [ ]:
def simulate_offset(offset):
    
    props = tile["output_observation_props"].clone()[:, :, -1:]
    props[0, 1, :] = offset
    tile_bw = deepcopy(tile)
    tile_bw["output_observation_props"] = props

    with torch.no_grad():
        y_pred = model(tile_bw)["output_observations"]
        y_pred = [tensor.expected_value().cpu().numpy().squeeze() for tensor in y_pred]

    plt.figure(figsize=(6.5, 5))
    plt.title(f"183 +/- {offset:.2f} GHz")
    plt.imshow(y_pred[0], vmin=240, vmax=270)
    plt.colorbar(label="$T_b$ [K]")
    plt.grid(False)

In [ ]:
interact(simulate_offset, offset=(1.0, 7.0))

## EIA

In [ ]:
def simulate_eia(eia):
    
    props = tile["output_observation_props"].clone()[:, :, [-5]]
    props[0, -2, :] += eia
    tile_bw = deepcopy(tile)
    tile_bw["output_observation_props"] = props

    with torch.no_grad():
        y_pred = model(tile_bw)["output_observations"]
        y_pred = [tensor.expected_value().cpu().numpy().squeeze() for tensor in y_pred]

    plt.figure(figsize=(6.5, 5))
    plt.title(rf"183 +/- 7 GHz, $\Delta\theta = ${eia:.2f}")
    plt.imshow(y_pred[0], vmin=220, vmax=280)
    plt.colorbar(label="$T_b$ [K]")
    plt.grid(False)

In [ ]:
interact(simulate_eia, eia=(-20.0, 20.0))

In [ ]:
plt.imshow(props[0, -2, -1])
plt.colorbar()

## Run simul